# Image → Text → Summary → Sentiment using `langchain_huggingface`

## Objective

This notebook demonstrates a similar workflow using the Hugging Face ecosystem together with **`langchain_huggingface`**.

We will:

1. Load the same image.
2. Use a Hugging Face vision-language pipeline to read the text from the image.
3. Wrap a local Hugging Face text model with `langchain_huggingface.HuggingFacePipeline`.
4. Use LangChain prompt templates for summarization.
5. Use another LangChain prompt for sentiment analysis.
6. Combine the results.

### Why is the image-extraction step different?

`ChatOpenAI` directly accepts image content through its LangChain chat interface.

`langchain_huggingface.HuggingFacePipeline` is primarily a LangChain wrapper around **text-generation/text-to-text Transformers pipelines**. Therefore, in this notebook:

```text
Transformers vision pipeline → image text extraction
            ↓
langchain_huggingface.HuggingFacePipeline
            ↓
Summary + Sentiment
```

The downstream language processing is done through LangChain's Hugging Face integration.

## Step 1 — Install libraries

The image model and local language model require PyTorch and Transformers.

> The first execution may download model weights from Hugging Face and can take time depending on your internet connection and hardware.

In [ ]:
# Uncomment and run once if needed.
# %pip install -U "transformers<5" torch pillow accelerate langchain-huggingface langchain-core sentencepiece pandas

## Step 2 — Import libraries

In [ ]:
from pathlib import Path

import pandas as pd
from PIL import Image
from IPython.display import display
from transformers import pipeline

from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

## Step 3 — Load and display the image

In [ ]:
IMAGE_PATH = Path("believe_in_yourself.png")

image = Image.open(IMAGE_PATH).convert("RGB")
print("Image size:", image.size)
display(image)

## Step 4 — Load a Hugging Face vision-language pipeline

A vision-language model can accept an image and a text instruction.

We ask it to behave like an OCR system and return all visible text.

The exact model can be replaced by another compatible Hugging Face image-text-to-text model if required.

> Vision-language models can require substantial RAM. For classroom machines with limited resources, use a smaller compatible VLM or run this section on a GPU-enabled environment.

In [ ]:
VISION_MODEL = "llava-hf/llava-interleave-qwen-0.5b-hf"

image_reader = pipeline(
    task="image-text-to-text",
    model=VISION_MODEL,
    device=-1
)

print("Loaded vision model:", VISION_MODEL)

## Step 5 — Ask the vision model to extract the visible text

The pipeline receives a conversation-style message containing:

- the local image;
- a text instruction asking for transcription.

We use `return_full_text=False` so the returned value focuses on the generated answer.

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": (
                    "Extract all visible written text from this image. "
                    "Keep the original reading order. "
                    "Do not summarize or explain. Return only the text."
                ),
            },
        ],
    }
]

vision_output = image_reader(
    text=messages,
    max_new_tokens=250,
    return_full_text=False
)

vision_output

## Step 6 — Store the extracted text

Pipeline output formats can vary slightly between compatible models.

The next cell extracts the generated string into a variable named `extracted_text`.

In [ ]:
result = vision_output[0]

generated = result.get("generated_text", "")

if isinstance(generated, list):
    # Some chat-style pipelines return message dictionaries.
    text_parts = []
    for item in generated:
        if isinstance(item, dict):
            content = item.get("content", "")
            if isinstance(content, str):
                text_parts.append(content)
    extracted_text = "\n".join(text_parts).strip()
else:
    extracted_text = str(generated).strip()

print("EXTRACTED TEXT")
print("=" * 70)
print(extracted_text)

## Step 7 — Load a text-to-text model through `langchain_huggingface`

Now we move from image processing to NLP.

`HuggingFacePipeline.from_model_id()` creates the Transformers model and then exposes it as a LangChain LLM.

We use FLAN-T5 because it can follow instructions such as:

- summarize this text;
- classify the sentiment.

In [ ]:
TEXT_MODEL = "google/flan-t5-base"

hf_llm = HuggingFacePipeline.from_model_id(
    model_id=TEXT_MODEL,
    task="text2text-generation",
    pipeline_kwargs={
        "max_new_tokens": 150,
        "do_sample": False
    }
)

print("Loaded LangChain Hugging Face model:", TEXT_MODEL)

## Step 8 — Build the summarization prompt

Unlike a dedicated summarization pipeline, this approach gives an instruction-following model a natural-language task.

LangChain's `PromptTemplate` fills `{text}` with the OCR output.

In [ ]:
summary_prompt = PromptTemplate.from_template(
    "Summarize the following motivational text in 2 concise sentences. "
    "Do not add information.\n\nText:\n{text}\n\nSummary:"
)

summary_chain = summary_prompt | hf_llm

## Step 9 — Generate the summary

In [ ]:
summary = summary_chain.invoke({"text": extracted_text})

print("SUMMARY")
print("=" * 70)
print(summary)

## Step 10 — Build a sentiment-analysis prompt

The same local Hugging Face LLM can be reused for a different task.

This demonstrates an important LLM idea:

```text
Same model + different prompt = different task
```

In [ ]:
sentiment_prompt = PromptTemplate.from_template(
    "Classify the sentiment of the following text as only "
    "Positive, Negative, or Neutral.\n\n"
    "Text:\n{text}\n\nSentiment:"
)

sentiment_chain = sentiment_prompt | hf_llm

## Step 11 — Run sentiment analysis

In [ ]:
sentiment = sentiment_chain.invoke({"text": extracted_text}).strip()

print("SENTIMENT")
print("=" * 70)
print(sentiment)

## Step 12 — Produce a final DataFrame

In [ ]:
report = pd.DataFrame([{
    "image": IMAGE_PATH.name,
    "extracted_text": extracted_text,
    "summary": summary,
    "sentiment": sentiment
}])

report

## Step 13 — Save the result

In [ ]:
OUTPUT_FILE = "langchain_huggingface_image_text_analysis.csv"
report.to_csv(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE)

## Behind the scenes

### Vision stage
The image-text model converts visual information into generated text. It is performing a vision-language generation task rather than using a traditional OCR engine such as Tesseract.

### LangChain Hugging Face stage
`HuggingFacePipeline` adapts a local Transformers text-generation pipeline to LangChain's Runnable interface.

That allows this syntax:

```text
PromptTemplate | HuggingFacePipeline
```

### Local execution
Unlike the OpenAI notebook, the Hugging Face models can run locally after the model files are downloaded. The trade-off is that local inference may need significantly more RAM/GPU resources.